In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

In [ ]:
train["target"] = (train["Heart Disease"] == "Presence").astype(int)
train.drop(columns=["Heart Disease"], inplace=True)


In [ ]:
X = train.drop(columns=["id", "target"])
y = train["target"]
X_test = test.drop(columns=["id"])


In [ ]:
X["HR_per_age"] = X["Max HR"] / (X["Age"] + 1)
X_test["HR_per_age"] = X_test["Max HR"] / (X_test["Age"] + 1)

X["Chol_per_age"] = X["Cholesterol"] / (X["Age"] + 1)
X_test["Chol_per_age"] = X_test["Cholesterol"] / (X_test["Age"] + 1)

X["BP_Chol"] = X["BP"] * X["Cholesterol"]
X_test["BP_Chol"] = X_test["BP"] * X_test["Cholesterol"]

X["ST_severe"] = (X["ST depression"] > 2).astype(int)
X_test["ST_severe"] = (X_test["ST depression"] > 2).astype(int)


In [ ]:
X.columns = X.columns.str.replace(" ", "_")
X_test.columns = X_test.columns.str.replace(" ", "_")


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))


In [ ]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"Fold {fold+1}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / 5


In [ ]:
print("CV Log Loss:", log_loss(y, oof_preds))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import log_loss

xgb_oof = np.zeros(len(X))
xgb_test = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"XGB Fold {fold+1}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = XGBClassifier(
        n_estimators=1500,
        learning_rate=0.02,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        tree_method="hist"
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    xgb_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    xgb_test += model.predict_proba(X_test)[:, 1] / 5

print("XGBoost CV Log Loss:", log_loss(y, xgb_oof))


In [ ]:
cat_features = [
    "Sex",
    "Chest_pain_type",
    "FBS_over_120",
    "EKG_results",
    "Exercise_angina",
    "Slope_of_ST",
    "Number_of_vessels_fluro",
    "Thallium"
]


In [ ]:
cat_oof = np.zeros(len(X))
cat_test = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"CatBoost Fold {fold+1}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.02,
        depth=6,
        eval_metric="Logloss",
        random_state=42,
        verbose=0
    )
    
    model.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val)
    )
    
    cat_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    cat_test += model.predict_proba(X_test)[:, 1] / 5

print("CatBoost CV Log Loss:", log_loss(y, cat_oof))


In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "Heart Disease": test_preds
})

submission.to_csv("submission.csv", index=False)
